<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi-e2/notebooks/gsm8k/E0_Experiment_Entropy_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Experiment 0: Search-Free LLaDA Entropy Decoding (Lowest-Predictive-Entropy Position Selection at T=0) on GSM8K — Test Split (200 Primary + 300 Extended Questions)

This notebook runs **Experiment 0 (E0)** on the predetermined GSM8K `test` split with `LLaDA-8B-Instruct` using search-free **Entropy Decoding (`entropy_t0`, 256 NFEs, $T=0$)** under the shared two-shot decomposition protocol (`gsm8k_shared_v2`):
At each of the 256 unmasking steps ($T=0$, 0 search NFEs, 256 completion NFEs), for current masked state $s_t$ with remaining masked positions $\mathcal{M}(s_t)$, each masked position $j \in \mathcal{M}(s_t)$ is scored by its predictive Shannon entropy $h_j(s_t)$ over non-mask vocabulary tokens, and the position $j^*$ with the **lowest predictive entropy** commits its greedy token $\hat{v}_{j^*} = \arg\max_{v \neq [\text{MASK}]} p_\theta(v \mid j^*, s_t)$:
$$
h_j(s_t) = -\sum_{v \neq [\text{MASK}]} p_\theta(v \mid j, s_t) \log p_\theta(v \mid j, s_t), \qquad j^* = \arg\min_{j \in \mathcal{M}(s_t)} h_j(s_t)
$$

### Test Dataset Split Organization (Aligned with E1 & E2 Test Runs):
1. **Primary 200 Test Samples** (`sample_slice=(0, 200)`): The exact 200 predetermined test questions from `configs/gsm8k/experiment_samples.json` (`sample_seed=1`).
2. **Extended 300 Test Samples** (`sample_slice=(200, 500)`): 300 additional held-out test questions from `configs/gsm8k/experiment_samples_500.json` under the identical `sample_seed=1` ordering.
3. **Combined 500 Test Samples**: Full 500-question test pool evaluation, Wilson 95% confidence intervals, and paired comparison.

### Protocol & Execution Features:
- **Cooperative Varlen FlashAttention-2 Batching**: Packs active states across questions in each batch (`BATCH_SIZE = 2` on L4, `4` or `8` on A100) with 100% bit-for-bit numerical parity with serial decoding.
- **Incremental Google Drive Persistence & Resumption**: Saves per-question artifacts (`metrics.json`, `completion.json`, `trace.jsonl`, `complete.json`) and summary CSVs after every batch so interrupted Colab sessions resume automatically with 0 redundant forward passes.


## 1. Verify GPU Hardware
Verify CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name} | Total VRAM: {vram_gb:.2f} GiB')
if vram_gb < 20.0:
    raise RuntimeError('This notebook uses unquantized BF16 weights. Select an L4 or A100 GPU runtime.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('This GPU does not support bfloat16 inference.')
print('✓ GPU hardware verification passed.')


## 2. Repository Setup
Clone or update the GitHub repository on branch `adi-e2`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/CooL-Legend/CSCI544-DLM.git'
BRANCH = 'adi-e2'
REPO_DIR = Path('/content/CSCI544-DLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        token = os.environ.get('GITHUB_TOKEN', '')
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                pass
        clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
        subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Working directory: {Path.cwd()}')


## 3. Install Dependencies
Install pinned dependencies and editable package.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/llada-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
    import importlib
    importlib.invalidate_caches()
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
print('Dependencies installed successfully.')


## 4. Run Algorithmic Pre-Flight Unit Tests
Verify E0 Entropy test-split alignment with E1/E2 (200 primary and 500 extended manifests), batched decoding parity, and checkpoint resumption before loading weights.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/tasks/test_e0_test_stage.py', '-v'],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Pre-flight E0 test stage unit tests failed!')
print('All E0 algorithmic unit tests passed.')


## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent artifact and model caching.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/CSCI544-DLM')
OUTPUT_DIR = Path('results/e0_entropy_test')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_ROOT / 'results/e0_entropy_test'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = DRIVE_ROOT / 'cache/huggingface'
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

print(f'Output directory: {OUTPUT_DIR.resolve()}')


## 6. Experiment Controls & Staged Model Loading
Configure search-free E0 Entropy decoding (`entropy_t0`, 256 steps at $T=0$), cooperative batch size 2, and load `LLaDA-8B-Instruct` from staged NVMe storage.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_0 import context, load_model

STAGE = 'test'
METHODS = ('entropy',)
# On NVIDIA L4 (22.4 GiB VRAM), BATCH_SIZE = 2 is recommended for stability;
# BATCH_SIZE = 4 or 8 is suited for A100 (40/80 GB) or High-RAM runtimes.
BATCH_SIZE = 2
RESUME = True

# Load base context on the 500-question test split to initialize model identity
base_ctx = context(stage=STAGE, methods=METHODS, sample_slice=(0, 500))

profile = get_profile('llada')
model_name = profile['repo_id'].split('/')[-1]
drive_model_dir = DRIVE_ROOT / 'models' / model_name
runtime_model_dir = Path('/content/models') / model_name

# Fast local NVMe cache in Colab runtime if available on Drive (or cache to Drive first)
if IN_COLAB:
    if not (drive_model_dir / 'config.json').is_file() and not (runtime_model_dir / 'config.json').is_file():
        subprocess.run([
            sys.executable, '-u', str(REPO_DIR / 'scripts/cache_model.py'),
            '--model', 'llada', '--drive-dir', str(drive_model_dir),
            '--runtime-dir', str(runtime_model_dir),
        ], check=True)
    elif drive_model_dir.is_dir() and not (runtime_model_dir / 'config.json').is_file():
        print(f'Copying model from Google Drive to local fast runtime storage: {runtime_model_dir}...')
        shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = str(runtime_model_dir if runtime_model_dir.is_dir() else (drive_model_dir if drive_model_dir.is_dir() else profile['repo_id']))
print('Using model location:', MODEL_PATH)
print('Loading LLaDA-8B-Instruct wrapper on GPU in bfloat16...')
wrapper = load_model(base_ctx, model_path=MODEL_PATH, device='cuda', local_files_only=Path(MODEL_PATH).is_dir())
print(f'✓ Model loaded: {wrapper.identity["family"]} ({wrapper.identity["repo_id"]} @ {wrapper.identity["revision"]})')
print(f'✓ Stage: {STAGE} | Method: {METHODS[0]} (entropy_t0) | Total Test Pool: {len(base_ctx["examples"])} questions | Batch Size: {BATCH_SIZE}')


## 7. Execute Part 1: Primary Test Set (Original 200 Test Questions)
Evaluates the exact 200 predetermined test questions (`sample_slice=(0, 200)`) from `configs/gsm8k/experiment_samples.json`.

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_0 import run_e0_stage

ctx_200 = context(stage=STAGE, methods=METHODS, sample_slice=(0, 200))
ctx_200['setup'] = dict(base_ctx.get('setup', {}))
print('Starting Part 1 (Primary 200 Test Questions) with E0 Entropy decoding...')
t0 = perf_counter()
q_rows_200, agg_rows_200 = run_e0_stage(
    wrapper,
    ctx_200,
    OUTPUT_DIR / 'primary_200',
    methods=METHODS,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 1 Complete in {(perf_counter() - t0) / 60:.2f} minutes ({len(q_rows_200)} questions).')


## 8. Execute Part 2: Extended Test Set (Additional 300 Test Questions)
Evaluates the additional 300 predetermined test questions (`sample_slice=(200, 500)`) from `configs/gsm8k/experiment_samples_500.json`.

In [ ]:
ctx_300 = context(stage=STAGE, methods=METHODS, sample_slice=(200, 500))
ctx_300['setup'] = dict(base_ctx.get('setup', {}))
print('Starting Part 2 (Extended 300 Test Questions) with E0 Entropy decoding...')
t0 = perf_counter()
q_rows_300, agg_rows_300 = run_e0_stage(
    wrapper,
    ctx_300,
    OUTPUT_DIR / 'extended_300',
    methods=METHODS,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 2 Complete in {(perf_counter() - t0) / 60:.2f} minutes ({len(q_rows_300)} questions).')


## 9. Results & Summary Metrics (Primary 200, Extended 300, and Combined 500 Test Questions)
Display aggregate accuracy, Wilson 95% confidence intervals, parse rate, NFE counts, and wall-clock latency across Primary 200, Extended 300, and Combined 500 test questions.

In [ ]:
import pandas as pd
from IPython.display import display
from dlm_project.experiments.sweep_report import write_csv
from dlm_project.tasks.gsm8k.experiment_0 import (
    compute_e0_aggregate_rows,
    compute_paired_comparison,
)

summaries_dir = OUTPUT_DIR / 'summaries'
summaries_dir.mkdir(parents=True, exist_ok=True)

agg_cols = [
    'method', 'condition', 'sample_count', 'correct_count', 'accuracy_percent',
    'ci_95_low', 'ci_95_high', 'parse_rate', 'mean_search_nfe',
    'mean_completion_nfe', 'mean_total_nfe', 'mean_completion_wall_seconds',
]

combined_q_rows = q_rows_200 + q_rows_300
combined_agg_rows = compute_e0_aggregate_rows(combined_q_rows, STAGE, METHODS)

for title, rows, agg, slug in [
    ('Part 1: Primary 200 Test Questions', q_rows_200, agg_rows_200, 'primary_200'),
    ('Part 2: Extended 300 Test Questions', q_rows_300, agg_rows_300, 'extended_300'),
    ('Combined Pool: All 500 Test Questions', combined_q_rows, combined_agg_rows, 'combined_500'),
]:
    if rows:
        write_csv(summaries_dir / f'e0_entropy_{slug}_per_question.csv', rows)
    if agg:
        write_csv(summaries_dir / f'e0_entropy_{slug}_aggregate.csv', agg)
        df_agg = pd.DataFrame(agg)
        print(f'=== {title} Aggregate Metrics ===')
        display(df_agg[[c for c in agg_cols if c in df_agg.columns]])

if combined_q_rows:
    write_csv(summaries_dir / 'e0_test_per_question.csv', combined_q_rows)
if combined_agg_rows:
    write_csv(summaries_dir / 'e0_test_aggregate.csv', combined_agg_rows)

# Optional cross-baseline paired comparison if companion E0 test notebook results exist on Drive
companion_pq = (DRIVE_ROOT if IN_COLAB else Path('.')) / 'results/e0_confidence_test' / 'summaries/e0_test_per_question.csv'
if companion_pq.is_file() and combined_q_rows:
    comp_rows = pd.read_csv(companion_pq).to_dict('records')
    paired = compute_paired_comparison(combined_q_rows + comp_rows, baseline_method='confidence', comparison_method='entropy')
    if paired:
        print('=== Paired Comparison (E0 Confidence vs. E0 Entropy on Common Test Questions) ===')
        display(pd.DataFrame(paired))
        write_csv(summaries_dir / 'e0_test_paired_comparison.csv', paired)

if combined_q_rows:
    df_pq = pd.DataFrame(combined_q_rows)
    pq_cols = [
        'example_id', 'method', 'condition', 'exact_match', 'parse_success',
        'predicted_answer', 'reference_answer', 'completion_nfes', 'total_nfes',
        'first_terminator_position', 'completion_wall_seconds',
    ]
    print('\n=== Sample Per-Question Test Outcomes (First 20 Questions) ===')
    display(df_pq[[c for c in pq_cols if c in df_pq.columns]].head(20))


## 10. Interactive Test Question & Unmasking Trace Inspector
Inspect the decoded reasoning steps, first EOS/EOT termination boundary, extracted numerical answer, and unmasking commit trace for any evaluated test question.

In [ ]:
import json
from IPython.display import Code, display

INSPECT_SPLIT = 'primary_200'  # 'primary_200' or 'extended_300'
INSPECT_QUESTION_IDX = 0

split_ctx = ctx_200 if INSPECT_SPLIT == 'primary_200' else ctx_300
sample_ex = split_ctx['examples'][INSPECT_QUESTION_IDX]
sample_ex_id = sample_ex['example_id']
q_dir = OUTPUT_DIR / INSPECT_SPLIT / STAGE / 'entropy' / sample_ex_id
metrics_json = q_dir / 'metrics.json'
trace_jsonl = q_dir / 'trace.jsonl'

if metrics_json.is_file():
    met = json.loads(metrics_json.read_text())
    print(f'=== E0 Entropy Test Outcome ({sample_ex_id}) ===')
    print('Question:\n', met['question'])
    print(f'Exact Match: {met["exact_match"]} | Predicted: {met["predicted_answer"]} | Reference: {met["reference_answer"]}')
    print(f'NFEs (search/completion/total): {met["search_nfes"]}/{met["completion_nfes"]}/{met["total_nfes"]} | First EOS/EOT Pos: {met["first_terminator_position"]}')
    print('\nScored Prefix (before first EOS/EOT):')
    display(Code(met['scoring_text'], language='text'))

if trace_jsonl.is_file():
    commits = [json.loads(line) for line in trace_jsonl.read_text().splitlines() if line.strip() and '"stage": "commit"' in line]
    print(f'\nUnmasking Commit Trace ({len(commits)} steps recorded; showing first 3 commits):')
    for event in commits[:3]:
        print(json.dumps(event, indent=2))


## 11. Synchronize All Artifacts to Google Drive
Flush and sync all in-flight buffers to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed successfully.')
